# 01 - HEC-RAS Workflow

Point cloud to hydraulics for the Ayase River (綾瀬川), Saitama.

Runs the whole chain: discover published LiDAR tiles, extract cross-sections,
write a HEC-RAS project, compute it, and read depth and velocity back out.
Needs **HEC-RAS 7.x on Windows** - see `docs/HECRAS_GUIDE.md`.

Data: 埼玉県 河川点群データ (CC BY 4.0), acquired by UAV **and narrow multibeam
echosounder**, so it includes the submerged bed.


In [ ]:
import sys; sys.path.insert(0, '../src')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

from aquanexus.data.pointcloud import (load_tile_index, tiles_for_river,
                                       download_tile, extract_las, load_points)
from aquanexus.hecras.geometry import extract_sections, to_hecras_geometry
from aquanexus.hecras.project import write_project, SteadyFlowProfile
from aquanexus.hecras.runner import RasController, find_hecras_exe
from aquanexus.data.validator import validate_sections

print('HEC-RAS:', find_hecras_exe())


## 1. Tile index

The index is not published as a file - it exists only inside the Mapbox vector
tiles behind the prefecture's web map. `load_tile_index()` decodes those and
caches the result.


In [ ]:
tiles = load_tile_index()
ayase = tiles_for_river('ayasegawa', tiles)
print(f'{len(tiles)} tiles across {len({t.river for t in tiles})} rivers')
print(f'Ayase: {len(ayase)} tiles')
pd.DataFrame([{'mesh': t.mesh, 'lat': t.lat, 'lon': t.lon} for t in ayase]).head()


### Coverage

Both Ayase and Naka are fully contiguous - no gaps over 1 km. Ayase is the
primary reach: 3.4x smaller download and a longer continuous extent.


In [ ]:
import math

def haversine(a, b):
    R = 6371000.0
    p1, p2 = math.radians(a.lat), math.radians(b.lat)
    dp, dl = p2 - p1, math.radians(b.lon - a.lon)
    h = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(h))

steps = [haversine(ayase[i], ayase[i+1]) for i in range(len(ayase)-1)]
print(f'chain {sum(steps)/1000:.1f} km | median step {np.median(steps):.0f} m'
      f' | max {max(steps):.0f} m')

fig, ax = plt.subplots(figsize=(6, 7))
ax.plot([t.lon for t in ayase], [t.lat for t in ayase], '.-', ms=4)
ax.set_xlabel('longitude'); ax.set_ylabel('latitude')
ax.set_title('Ayase tile centroids'); ax.set_aspect('equal')
plt.show()


## 2. One tile

Tiles average ~107 MB zipped and carry roughly 27 points/m². `thin` samples
every n-th point, ample for cross-sections.


In [ ]:
tile = next(t for t in ayase if t.mesh == 'ayasegawa-0610')
points = load_points(extract_las(download_tile(tile)), thin=4)
print(f'{len(points):,} points')
print(f'extent {np.ptp(points[:,0]):.0f} x {np.ptp(points[:,1]):.0f} m, '
      f'elevation {points[:,2].min():.2f}..{points[:,2].max():.2f} m')


In [ ]:
# Colour by elevation: the channel shows as the low band.
s = points[::10]
fig, ax = plt.subplots(figsize=(7, 8))
sc = ax.scatter(s[:,0], s[:,1], c=s[:,2], s=1, cmap='terrain')
plt.colorbar(sc, label='elevation (m)')
ax.set_aspect('equal'); ax.set_title('ayasegawa-0610')
plt.show()


## 3. Cross-sections

The tiles carry **no ground classification** - every point is class 1 - so bare
earth cannot be selected by filtering. Points are binned across the section and a
low elevation quantile taken per bin, keeping the bed while rejecting canopy
above and multibeam outliers below.


In [ ]:
# Channel axis from the lowest 15% of points.
low = points[points[:,2] <= np.quantile(points[:,2], 0.15)]
centre = low[:,:2].mean(axis=0)
_, _, vt = np.linalg.svd(low[:,:2] - centre, full_matrices=False)
tangent = vt[0]
centreline = np.array([centre - tangent*150, centre, centre + tangent*150])

sections = extract_sections(points, centreline, spacing=50.0,
                            half_width=80.0, smooth_window=1)
print(f'{len(sections)} sections')


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for xs in sections:
    ax.plot(xs.station, xs.elevation, lw=1, label=f'RS {xs.river_station:.0f}')
ax.set_xlabel('station (m)'); ax.set_ylabel('bed elevation (m)')
ax.set_title('Extracted cross-sections'); ax.legend(fontsize=7)
plt.show()


### Validate before building geometry

A section cut through a bridge pier or a constriction comes out far narrower than
its neighbours and silently distorts velocity. The validator flags that.


In [ ]:
print(validate_sections(sections))


## 4. Build and run the model

Discharges are the 1st percentile, median and 99th percentile of the **observed**
Ayase record, so the sweep spans conditions the river actually experiences.


In [ ]:
profiles = [SteadyFlowProfile('Low', 0.25),
            SteadyFlowProfile('Median', 9.7),
            SteadyFlowProfile('High', 64.2)]

prj = write_project('../data/hecras/notebook_demo', 'Ayase',
                    to_hecras_geometry(sections, river='Ayase', reach='Main'),
                    profiles, river='Ayase', reach='Main',
                    upstream_station=max(s.river_station for s in sections))
print(prj)


In [ ]:
with RasController(prj) as ras:
    ok, messages = ras.compute()
    print('compute:', 'SUCCESS' if ok else 'FAILED')
    results = ({p.name: pd.DataFrame(ras.profile_results(profile=i))
                for i, p in enumerate(profiles, 1)} if ok else {})

# If this fails, read <project>.<plan>.computeMsgs.txt - the COM API returns
# only a generic summary while the per-station reasons are written to disk.
results['Median'] if results else messages


In [ ]:
if results:
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
    for ax, col, label in zip(axes, ['depth', 'velocity', 'top_width'],
                              ['depth (m)', 'velocity (m/s)', 'top width (m)']):
        for name, df in results.items():
            ax.plot(df.river_station, df[col], 'o-', label=name)
        ax.set_xlabel('river station (m)'); ax.set_ylabel(label)
        ax.invert_xaxis()
    axes[0].legend(); plt.tight_layout(); plt.show()


### What to notice

At the high profile the top width jumps sharply on some sections - water
spreading onto the flat terrace beside the channel. That is a real flood terrace
(高水敷), confirmed by the fact that it conveys flow.

**Caveat:** Manning's n is assumed (0.035 channel, 0.06 overbank), not calibrated
- there is no gauged rating curve for this reach. Relative behaviour across the
flow sweep is more trustworthy than any single absolute depth.
